# Metacritic Game Site Web Scrape - Exploration/Learning

In this notebook I explore and experiment to build a web scraper that will reliably search through the game sites provided in the data set to acquire additional information.

## Planning

We'll try it out on one game before scaling it up to multiple and then the whole data set.

**Test Game**  
Baldur's Gate 3  

**URL**  
https://www.metacritic.com/game/baldurs-gate-3/  

**Which info do we want?**  
- Data from PS5 and XBox Series X platforms
- userscore, positive/negative/mixed/total user review counts

**Note**  
- 

In [122]:
import time
from urllib.parse import parse_qs, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup

from core.config import DATA_FORMATTED_PATH


## 1st attempt to extract a user score

In [119]:
# load formatted data set and extract platforms for which userscore is missing

df = pd.read_csv(DATA_FORMATTED_PATH)

df_bg3 = df.loc[df['title'] == "Baldur's Gate 3", :]

platforms = df_bg3['platform'].loc[df_bg3['userscore'].isna()]

In [81]:
# prepare example url and header
game_url = 'https://www.metacritic.com/game/baldurs-gate-3'

type_suffix = '/user-reviews'
platform_suffix = '?platform=playstation-5'

full_url = game_url + type_suffix + platform_suffix

headers = {
    "User-Agent": "Mozilla/5.0"
}


In [82]:
# try simple request
response = requests.get(full_url, headers=headers)
response.raise_for_status()

In [83]:
# try and get the user score from the ps5 page
html = response.text
soup = BeautifulSoup(html, 'html.parser')

# score matches
score_div = soup.find_all('div', class_ = "score-card-left__score-number")
score_num = score_div[0].find('span').get_text(strip = True)
print(score_num)

8.7


- First attempt at requesting from metacritic page was successful.
- user score was succesfully extracted for the PS5 version of BG3, which has been missing in the data so far

**Insights**  
- A way to extract the prefixes for all possible platforms (as defined by the data) is needed
    - Do a pre-scrape, so to speak, from pages that cover all possible platforms and extract the "slugs" for the platforms ???
    - Create a dict (in the config?) with the mappings to be used in the actual scrape

## Extract platform slugs

In [116]:
# request the main game page content
response_game = requests.get(game_url, headers = headers)
print(response_game)

<Response [200]>


In [118]:
# get the section with the platform list
html = response_game.text
soup = BeautifulSoup(html, 'html.parser')

platforms_list = soup.find_all('div', class_ = 'game-platforms__list')

platform_links = platforms_list[0].find_all(class_ = 'product-score-card--platform')

platform_mapping = {}
for link in platform_links:
    # Extract platform category
    pf_element = link.find('span', class_ = 'game-platform-logo__icon')
    pf_name = pf_element.get('title')
    
    # Extract corresponding slug in the url
    href = link.get('href')
    query = urlparse(href).query
    pf_slug = parse_qs(query)['platform'][0]

    # map them in a dict
    platform_mapping[pf_name] = pf_slug

platform_mapping

{'PC': 'pc',
 'PlayStation 5': 'playstation-5',
 'Xbox Series X': 'xbox-series-x'}

## Trial: Iteratively scraping User Score

Use platform slug mapping to iteratively extract user score for BG3 for PS5 and XBox, which have been missing from the data

In [123]:
missing_slugs = platforms.map(platform_mapping).reset_index(drop = True)

for platform, slug in zip(platforms, missing_slugs):
    missing_url = game_url + type_suffix + '/?platform=' + slug
    print(missing_url)

    response = requests.get(missing_url, headers=headers)
    response.raise_for_status()

    lag = response.elapsed.total_seconds()
    
    html = response.text
    soup = BeautifulSoup(html, 'html.parser')

    score_div = soup.find_all('div', class_ = "score-card-left__score-number")
    score_str = score_div[0].find('span').get_text(strip = True)

    print(f'{platform}: {score_str}')

    if score_str != 'tbd':
        mask = df_bg3['platform'] == platform
        df_bg3.loc[mask, 'userscore'] = float(score_str)
    
    time.sleep(5*lag)

https://www.metacritic.com/game/baldurs-gate-3/user-reviews/?platform=playstation-5
PlayStation 5: 8.7
https://www.metacritic.com/game/baldurs-gate-3/user-reviews/?platform=xbox-series-x
Xbox Series X: 7.1
